In [19]:
import sys
from pathlib import Path

import os
import requests
from dotenv import load_dotenv

import warnings
# on va ignorer les messages avertissements a cause d'un RAGASS pas compatible mystral v2.0
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [20]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

load_dotenv()

TOKEN = os.getenv("API_KEY")
TOKEN_MISTRAL = os.getenv("MISTRAL_API_KEY")

API_URL_ASK = "http://127.0.0.1:8000/ask"
API_URL_ASK_DEBUG = "http://127.0.0.1:8000/ask/debug"


In [21]:
#appel de mon api (probleme de versionnage de mistral)
question = "Quels événements culturels gratuits ont lieu à Montpellier ?"

resp = requests.post(
    API_URL_ASK,
    json={"question": question},
    headers={"x-api-key": TOKEN},
    timeout=60,
)

print(resp.status_code)
print(resp.json())

200
{'question': 'Quels événements culturels gratuits ont lieu à Montpellier ?', 'answer': 'Titre : Visite libre de la bibliothèque Jean Claparède\n\nLieu : Musée Fabre, Montpellier\nDate : 20 et 21 septembre 2025\nDescription : Visite libre de la bibliothèque Jean Claparède avec une lecture en liberté autour de l\'architecture et de la poésie occitane.\n\n---\nPourquoi cet événement pourrait vous intéresser :\nSi vous aimez la poésie ou l\'architecture, cette visite libre vous permettra de découvrir un lieu culturel emblématique de Montpellier dans un cadre poétique et occitan.\n\n---\n\nTitre : Conférences, expositions et dédicaces de livres au Cercle Culturel Languedocien !\n\nLieu : Cercle Culturel Languedocien, Montpellier\nDate : 20 et 21 septembre 2025\nDescription : Conférences, expositions et dédicaces de livres autour de la culture languedocienne.\n\n---\nPourquoi cet événement pourrait vous intéresser :\nSi vous êtes passionné(e) par la culture régionale ou la littérature, c

In [22]:
# question
question = "Quels événements culturels gratuits ont lieu à Montpellier ?"

resp = requests.post(
    API_URL_ASK,
    json={"question": question},
    headers={"x-api-key": TOKEN},
    timeout=60,
)

print(resp.status_code)
data = resp.json()
print(data["answer"])

200
Titre : Visite libre de la bibliothèque Jean Claparède

Lieu : Musée Fabre, Montpellier
Date : 20 et 21 septembre 2025
Description : Visite libre de la bibliothèque Jean Claparède avec une lecture en liberté autour de l'architecture et de la poésie occitane.

---
Pourquoi cet événement pourrait vous intéresser :
Si vous aimez la poésie ou l'architecture, cette visite libre vous permettra de découvrir un lieu culturel emblématique de Montpellier dans un cadre poétique et occitan.

---

Titre : Conférences, expositions et dédicaces de livres au Cercle Culturel Languedocien !

Lieu : Cercle Culturel Languedocien, Montpellier
Date : 20 et 21 septembre 2025
Description : Conférences, expositions et dédicaces de livres autour de la culture languedocienne.

---
Pourquoi cet événement pourrait vous intéresser :
Si vous êtes passionné(e) par la culture régionale ou la littérature, cet événement propose des échanges enrichissants avec des auteurs et des expositions dédiées au Languedoc.

---

In [23]:
import requests

def run_rag_for_eval(question: str) -> dict:
    resp = requests.post(
        API_URL_ASK_DEBUG,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS:", resp.status_code)
    print("TEXT:", resp.text)

    try:
        data = resp.json()
    except Exception:
        data = {"raw_text": resp.text}

    if resp.status_code != 200:
        return {
            "question": question,
            "error": data,
            "answer": None,
            "contexts": None,
        }

    return {
        "question": question,
        "answer": data["answer"],
        "contexts": data["contexts"],
    }

In [24]:
eval_questions = [

{
    "question": "Y a-t-il des expositions à Montpellier ?",
    "ground_truth": (
        "À Montpellier, les expositions présentes dans le corpus sont : "
        "L'association Miss'terre fait son expo d'été les 27 et 28 juin 2025 ; "
        "Exposition 'Montpellier, regarder la ville autrement. Archives et architecture' "
        "aux Archives de Montpellier le 20 septembre 2025 ; "
        "Exposition 'L'Europe et son patrimoine' à la Maison de l'Europe "
        "de Montpellier les 20 et 21 septembre 2025."
    )
},

{
    "question": "Quels événements ont lieu au Musée Fabre ?",
    "ground_truth": (
        "Au Musée Fabre à Montpellier, un événement présent dans le corpus est : "
        "la visite libre de la bibliothèque Jean Claparède les 20 et 21 septembre 2025, "
        "avec une lecture autour de l'architecture et de la poésie occitane."
    )
},

{
    "question": "Quels événements ont lieu à Montpellier en septembre 2025 ?",
    "ground_truth": (
        "À Montpellier en septembre 2025, les événements présents dans le corpus sont : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre les 20 et 21 septembre ; "
        "les conférences, expositions et dédicaces au Cercle Culturel Languedocien les 20 et 21 septembre ; "
        "l'exposition 'Montpellier, regarder la ville autrement' aux Archives de Montpellier le 20 septembre ; "
        "et l'exposition 'L'Europe et son patrimoine' les 20 et 21 septembre."
    )
},

{
    "question": "Quels événements ont lieu aux Archives de Montpellier ?",
    "ground_truth": (
        "Aux Archives de Montpellier, l'événement présent dans le corpus est : "
        "l'exposition 'Montpellier, regarder la ville autrement. Archives et architecture', "
        "le 20 septembre 2025."
    )
},

{
    "question": "Quels événements culturels sont proposés à Montpellier ?",
    "ground_truth": (
        "Plusieurs événements culturels sont proposés à Montpellier dans le corpus : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
        "des conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
        "l'exposition 'Montpellier, regarder la ville autrement' aux Archives ; "
        "et l'exposition 'L'Europe et son patrimoine'."
    )
},

{
    "question": "Quels événements ont lieu le 20 septembre 2025 à Montpellier ?",
    "ground_truth": (
        "Le 20 septembre 2025 à Montpellier, les événements présents dans le corpus sont : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
        "les conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
        "et l'exposition 'Montpellier, regarder la ville autrement' aux Archives de Montpellier."
    )
},

{
    "question": "Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?",
    "ground_truth": (
        "Le week-end du 20 et 21 septembre 2025 à Montpellier comprend : "
        "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
        "les conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
        "et l'exposition 'L'Europe et son patrimoine'."
    )
},

{
    "question": "Y a-t-il des concerts de rock à Montpellier?",
    "ground_truth": (
        "Je ne trouve pas d'événement correspondant, je suis un assistant qui "
        "ne peut vous conseiller que des événements culturels."
    )
},

{
    "question": "Quels événements ont lieu à Paris ?",
    "ground_truth": (
        "Je ne trouve pas d'événement correspondant, je suis un assistant qui "
        "ne peut vous conseiller que des événements culturels."
    )
},

{
    "question": "Quels événements gratuits ont lieu à Montpellier ?",
    "ground_truth": (
        "Le corpus ne permet pas de confirmer explicitement la gratuité des événements, "
        "aucun événement ne peut donc être identifié avec certitude comme gratuit."
    )
}

]

In [25]:
rows = []

for item in eval_questions:
    result = run_rag_for_eval(item["question"])

    rows.append({
        "question": item["question"],
        "answer": result["answer"],
        "contexts": result["contexts"],
        "ground_truth": item["ground_truth"],
    })

rows[:1]

STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}
STATUS: 404
TEXT: {"detail":"Not Found"}


[{'question': 'Y a-t-il des expositions à Montpellier ?',
  'answer': None,
  'contexts': None,
  'ground_truth': "À Montpellier, les expositions présentes dans le corpus sont : L'association Miss'terre fait son expo d'été les 27 et 28 juin 2025 ; Exposition 'Montpellier, regarder la ville autrement. Archives et architecture' aux Archives de Montpellier le 20 septembre 2025 ; Exposition 'L'Europe et son patrimoine' à la Maison de l'Europe de Montpellier les 20 et 21 septembre 2025."}]

In [26]:
from datasets import Dataset

dataset = Dataset.from_list(rows)


In [27]:
for i in range(len(dataset)):
    row = dataset[i]

    print("=" * 100)
    print(f"EXEMPLE {i}")
    print("- Question :")
    print(row["question"])

    print("\n- Contextes récupérés :")
    for j, ctx in enumerate(row["contexts"], start=1):
        print(f"\n  [Contexte {j}]")
        print(ctx[:1000])  # coupe si trop long

    print("\n- Réponse générée :")
    print(row["answer"])

    if "ground_truth" in row:
        print("\n- Ground truth :")
        print(row["ground_truth"])

EXEMPLE 0
- Question :
Y a-t-il des expositions à Montpellier ?

- Contextes récupérés :


TypeError: 'NoneType' object is not iterable

In [ ]:
import pandas as pd

audit_rows = []

for i in range(len(dataset)):
    row = dataset[i]
    audit_rows.append({
        "id": i,
        "question": row["question"],
        "retrieval_ok": "",
        "answer_supported": "",
        "answer_complete": "",
        "hallucination": "",
        "commentaire": "",
    })

audit_df = pd.DataFrame(audit_rows)
audit_df

,id,question,retrieval_ok,answer_supported,answer_complete,hallucination,commentaire
0,0,Y a-t-il des expositions à Montpellier ?,,,,,
1,1,Quels événements ont lieu au Musée Fabre ?,,,,,
2,2,Quels événements ont lieu à Montpellier en sep...,,,,,
3,3,Quels événements ont lieu aux Archives de Mont...,,,,,
4,4,Quels événements culturels sont proposés à Mon...,,,,,
5,5,Quels événements ont lieu le 20 septembre 2025...,,,,,
6,6,Quels événements ont lieu le week-end du 20 au...,,,,,
7,7,Y a-t-il des concerts de rock à Montpellier?,,,,,
8,8,Quels événements ont lieu à Paris ?,,,,,
9,9,Quels événements gratuits ont lieu à Montpelli...,,,,,


In [ ]:
for i, row in enumerate(dataset):
    print("=" * 100)
    print(f"EXEMPLE {i}")

    print("\nQUESTION:")
    print(row["question"])

    print("\nCONTEXTES:")
    for j, ctx in enumerate(row["contexts"], start=1):
        print(f"\n--- Contexte {j} ---")
        print(ctx[:800])

    print("\nREPONSE:")
    print(row["answer"])

EXEMPLE 0

QUESTION:
Y a-t-il des expositions à Montpellier ?

CONTEXTES:

--- Contexte 1 ---
L'association Miss'terre fait son expo d'été !
Exposition
Association Miss'terre
Montpellier
Occitanie
2025-06-27
2025-06-28


--- Contexte 2 ---
Exposition "Montpellier, regarder la ville autrement. Archives et architecture"
À travers l'exemple de cinq monuments emblématiques de Montpellier, cette présentation de documents issus des fonds conservés aux Archives de Montpellier montrera comment l'architecture a changé de …
Archives de Montpellier
Montpellier
Occitanie
2025-09-20
2025-09-20


--- Contexte 3 ---
Exposition « L'Europe et son patrimoine »
Exposition : À la découverte du Label du patrimoine européen
Maison de l'Europe de Montpellier (Maison des relations internationales)
Montpellier
Occitanie
2025-09-20
2025-09-21


REPONSE:
Titre : L'association Miss'terre fait son expo d'été !

Lieu : Association Miss'terre, Montpellier
Date : 27-28 juin 2025
Description : Exposition organisée par

In [ ]:
for i, row in enumerate(dataset[:10]):
    assert isinstance(row["question"], str), f"question row {i} invalide"
    assert isinstance(row["answer"], str), f"answer row {i} invalide"
    assert isinstance(row["contexts"], list), f"contexts row {i} invalide"
    assert all(isinstance(c, str) for c in row["contexts"]), f"contexts row {i} contient autre chose que str"

print("Dataset OK sur les 10 premières lignes")

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import evaluate

llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=TOKEN_MISTRAL,
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[5]: HTTPStatusError(Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429})
Exception raised in Job[13]: HTTPStatusError(Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429})
Exception raised in Job[1]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')
Exception raised in Job[9]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')
Exception raised in Job[17]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')
Exception raised in Job[21]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')
Exception raised in Job[25]: TypeError(unsupported operand type(s) for +=: 'dict' and 'dict')
Exception raised in Job[37]: TypeError(un

{'faithfulness': 0.7343, 'answer_relevancy': nan, 'context_precision': 0.5500, 'context_recall': 0.5667}


In [ ]:
result_df = result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,Y a-t-il des expositions à Montpellier ?,[L'association Miss'terre fait son expo d'été ...,Titre : L'association Miss'terre fait son expo...,"À Montpellier, les expositions présentes dans ...",0.944444,NaN,0.5,1.000000
1,Quels événements ont lieu au Musée Fabre ?,"[Visite commentée : « Un musée, des bâtiments,...","Titre : Visite commentée : « Un musée, des bât...","Au Musée Fabre à Montpellier, un événement pré...",1.000000,NaN,0.0,0.000000
2,Quels événements ont lieu à Montpellier en sep...,[Visite guidée du cimetière protestant de Mont...,Titre : Visite guidée du cimetière protestant ...,"À Montpellier en septembre 2025, les événement...",1.000000,NaN,0.0,0.500000
3,Quels événements ont lieu aux Archives de Mont...,"[Exposition ""Montpellier, regarder la ville au...","Titre : Exposition ""Montpellier, regarder la v...","Aux Archives de Montpellier, l'événement prése...",1.000000,NaN,1.0,1.000000
4,Quels événements culturels sont proposés à Mon...,"[Exposition ""Montpellier, regarder la ville au...","Titre : Exposition ""Montpellier, regarder la v...",Plusieurs événements culturels sont proposés à...,0.937500,NaN,0.5,0.500000
5,Quels événements ont lieu le 20 septembre 2025...,[Visite guidée du cimetière protestant de Mont...,Titre : Visite guidée du cimetière protestant ...,"Le 20 septembre 2025 à Montpellier, les événem...",1.000000,NaN,0.5,0.333333
6,Quels événements ont lieu le week-end du 20 au...,"[Week-end ""Éclosion"" - 4 pratiques transformat...",Titre : Visite libre de la bibliothèque Jean C...,Le week-end du 20 et 21 septembre 2025 à Montp...,0.461538,NaN,0.0,0.333333
7,Y a-t-il des concerts de rock à Montpellier?,[Concert « Nocturnàlia »\n💃 Hommage à Max Rouq...,"Je ne trouve pas d'événement correspondant, je...","Je ne trouve pas d'événement correspondant, je...",0.000000,NaN,1.0,0.000000
8,Quels événements ont lieu à Paris ?,[Qui vive !\nPerformance | Séminaire | Cinéma ...,"Je ne trouve pas d'événement correspondant, je...","Je ne trouve pas d'événement correspondant, je...",0.000000,NaN,1.0,1.000000
9,Quels événements gratuits ont lieu à Montpelli...,[Visite libre de la bibliothèque Jean Claparèd...,Titre : Visite libre de la bibliothèque Jean C...,Le corpus ne permet pas de confirmer explicite...,1.000000,NaN,1.0,1.000000


In [ ]:
print(result)

{'faithfulness': 0.7343, 'answer_relevancy': nan, 'context_precision': 0.5500, 'context_recall': 0.5667}
